# Tren YOLO-seg (Colab)

**Forberedelse (én gang, gjøres lokalt på PC):**
```powershell
# Fra prosjektmappen:
Compress-Archive -Path data\annotated_seg -DestinationPath data_seg.zip
```
Last opp `data_seg.zip` til roten av Google Drive (samme som `data.zip`).

**Kjør cellene i rekkefølge:**

1. **Oppsett** — mount Drive, pakk ut data, klon kode, installer avhengigheter
2. **Treningsparametere** — juster om ønskelig
3. **Tren** — kjøres på GPU; early stopping stopper automatisk
4. **Last ned modell** — laster ned `best.pt` til PC

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

# --- 1. Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

MYDRIVE  = Path("/content/drive/MyDrive")
REPO_DIR = Path("/content/trash-bin-detection")
DATA_DIR = REPO_DIR / "data" / "annotated_seg"

# --- 2. Finn og pakk ut data_seg.zip ---
zip_path = MYDRIVE / "data_seg.zip"
if not zip_path.exists():
    raise FileNotFoundError(
        f"Finner ikke {zip_path}.\n"
        "Lag zip lokalt:  Compress-Archive -Path data\\annotated_seg -DestinationPath data_seg.zip\n"
        "Last den opp til roten av Google Drive."
    )

if not DATA_DIR.exists():
    print("Kopierer data_seg.zip til lokal lagring (raskere I/O under trening) ...")
    local_zip = Path("/content/data_seg.zip")
    shutil.copy2(zip_path, local_zip)
    print("Pakker ut ...")
    # zip inneholder annotated_seg/ — pakk ut til data/ slik at stien stemmer
    (REPO_DIR / "data").mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(local_zip), str(REPO_DIR / "data"))
    local_zip.unlink()
    print(f"Data klar: {DATA_DIR}")
else:
    print(f"Data finnes allerede: {DATA_DIR}")

# Vis datasett-størrelse
for split in ("train", "val", "test"):
    n = len(list((DATA_DIR / "labels" / split).glob("*.txt")))
    print(f"  {split}: {n} labels")

# --- 3. Klon kode fra GitHub ---
REPO_URL = "https://github.com/davgei/trash-bin-detection.git"
BRANCH   = "streetview-yolo-retry"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True
    )
    print(f"Klonet {BRANCH}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 4. Installer avhengigheter ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import torch
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'IKKE tilgjengelig — kjøres på CPU'}")
print("Oppsett OK.")

## Steg 2 — Treningsparametere

Standard er `yolo11n-seg` (nano, rask). Bytt til `yolo11s-seg` for litt bedre nøyaktighet.

In [ ]:
BASE_MODEL = "yolo11n-seg.pt"   # yolo11n-seg / yolo11s-seg / yolo11m-seg
EPOCHS     = 100
IMGSZ      = 640
BATCH      = 16                 # øk til 32 om GPU har nok minne
PATIENCE   = 30                 # stopper om ingen forbedring på 30 epoker
RUN_NAME   = "seg"

print(f"Modell : {BASE_MODEL}")
print(f"Epoker : {EPOCHS}  (early stop etter {PATIENCE} uten forbedring)")
print(f"Batch  : {BATCH}   Størrelse: {IMGSZ}px")

## Steg 3 — Tren

Følg med på `box_loss` og `seg_loss` — begge skal falle jevnt.  
Treningen stopper automatisk ved early stopping.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = Path("configs/data_seg.yaml")
OUT_DIR   = Path("models/trained")
OUT_DIR.mkdir(parents=True, exist_ok=True)

model   = YOLO(BASE_MODEL)
results = model.train(
    data     = str(DATA_YAML),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = BATCH,
    patience = PATIENCE,
    project  = str(OUT_DIR),
    name     = RUN_NAME,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nFerdig. Beste modell: {best_pt}")

## Steg 4 — Last ned modell

Laster ned `best.pt` til PC-en din.  
Legg den i `models/trained/` i prosjektet.

In [ ]:
from google.colab import files
import os

size_mb = os.path.getsize(best_pt) / 1024 / 1024
print(f"Laster ned {best_pt.name}  ({size_mb:.1f} MB) ...")
files.download(str(best_pt))
print("Ferdig.")